# WildlifeVision YOLO11 Trainer

This notebook trains the YOLO11 bird detector from the dataset exported by WildlifeVision Bird Library, then exports the browser-ready ONNX model and class labels.

**Before running:** In WildlifeVision → Bird Library, click **Train YOLO11 in Colab**. It opens this notebook and downloads `WildlifeVision-YOLO-dataset.zip`. Upload that ZIP in the next step.

In [ ]:
!git clone -b webgpu-yolo-training https://github.com/EdebN/curly-system.git
%cd /content/curly-system
!pip install -q -r training/requirements.txt

## 1. Upload the dataset ZIP

Choose the `WildlifeVision-YOLO-dataset.zip` file downloaded by Bird Library. This is the bridge between browser storage and Colab's Python runtime.

In [ ]:
from google.colab import files
from pathlib import Path

uploaded = files.upload()
zip_files = [name for name in uploaded if name.lower().endswith('.zip')]
if len(zip_files) != 1:
    raise RuntimeError(f'Upload exactly one WildlifeVision dataset ZIP. Found: {zip_files}')
dataset_zip = str(Path('/content') / zip_files[0])
print('Dataset received:', dataset_zip)

## 2. Train YOLO11 and export ONNX

Select **Runtime → Change runtime type → T4 GPU** first for faster training. Run the following cell once to display the button, then click **Train YOLO11**.


In [ ]:
import subprocess, sys, torch, ipywidgets as widgets
from IPython.display import display

train_button = widgets.Button(
    description='Train YOLO11',
    button_style='success',
    icon='play',
    tooltip='Start training and ONNX export'
)
train_output = widgets.Output(layout={'border': '1px solid #888', 'max_height': '420px', 'overflow_y': 'auto'})

def start_yolo_training(_):
    train_button.disabled = True
    train_button.description = 'Training…'
    with train_output:
        print('Preparing YOLO11 training…')
        device = '0' if torch.cuda.is_available() else 'cpu'
        print('Training device:', device)
        command = [
            sys.executable, 'training/train_yolo.py', dataset_zip,
            '--epochs', '100', '--imgsz', '640', '--batch', '-1',
            '--workers', '2', '--device', device,
            '--output', '/content/wildlifevision-training',
            '--name', 'bird-detector'
        ]
        try:
            process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
            for line in process.stdout:
                print(line, end='')
            exit_code = process.wait()
            if exit_code != 0:
                raise subprocess.CalledProcessError(exit_code, command)
            print('\nTraining and ONNX export completed. Continue to the download section below.')
            train_button.description = 'Training complete'
            train_button.button_style = 'success'
        except Exception as error:
            print('\nTraining failed:', error)
            train_button.description = 'Retry training'
            train_button.button_style = 'danger'
        finally:
            train_button.disabled = False

train_button.on_click(start_yolo_training)
display(train_button, train_output)

## 3. Download the trained detector

The ONNX file and labels JSON are the two files WildlifeVision needs. The `.pt` checkpoint is also offered for future fine-tuning.

In [ ]:
from google.colab import files
from pathlib import Path

weights = Path('/content/wildlifevision-training/runs/bird-detector/weights')
for filename in ['best.onnx', 'best.labels.json', 'best.pt']:
    path = weights / filename
    if not path.is_file():
        raise FileNotFoundError(f'Expected training output is missing: {path}')
    print('Downloading', filename)
    files.download(str(path))

## 4. Load the model in WildlifeVision

Open **Live camera** in WildlifeVision:
1. Choose `best.onnx` under **Choose YOLO11 model (.onnx)**.
2. Upload `best.labels.json` under **Upload labels JSON**.
3. Select **Load YOLO11**.

Keep `best.pt` if you plan to continue training later.